# One-step evaluation

Performance用データとcheckpointを確認してから1-step評価を実行し、保存済みmetricsとPNGを可視化します。YAMLはNotebookから変更しません。


## 1. 使用する設定


In [ ]:
from pathlib import Path
import csv, subprocess, sys
import matplotlib.pyplot as plt
import yaml
from IPython.display import Image, Markdown, display

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").is_dir() and (candidate / "performance").is_dir():
            return candidate
    raise FileNotFoundError("FNOV2リポジトリ内からNotebookを開いてください。")

def resolve(value, config_path):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (config_path.parent / path).resolve()

def status(ok, label, detail=""):
    mark = "✅" if ok else "❌"
    print(f"{mark} {label}" + (f": {detail}" if detail else ""))
    return ok

ROOT = find_project_root(Path.cwd())
print("project:", ROOT)
EXPERIMENT_ID = "ex001_baseline"
print("experiment:", EXPERIMENT_ID)
TEST_YAML = ROOT / "performance/configs/test.yaml"
test_cfg = yaml.safe_load(TEST_YAML.read_text(encoding="utf-8"))
loader_path = resolve(test_cfg["test_dataloader_config"], TEST_YAML)
loader_cfg = yaml.safe_load(loader_path.read_text(encoding="utf-8"))
kind = loader_cfg["evaluation"]["type"]
data_dir = resolve(loader_cfg["data"]["directory"], loader_path)
checkpoint = resolve(test_cfg["checkpoint"], TEST_YAML)
result_dir = resolve(test_cfg["output"]["root_directory"], TEST_YAML) / kind / "one_step"


## 2. 設定の可視化


In [ ]:
summary={
 "evaluation":kind, "data directory":str(data_dir), "test Re":loader_cfg["test_re"],
 "checkpoint":str(checkpoint), "device":test_cfg["device"], "output":str(result_dir),
 "dataset config":str(resolve(loader_cfg["dataset_config"],loader_path)),
}
print(yaml.safe_dump(summary,allow_unicode=True,sort_keys=False))
display(Markdown("### test.yaml")); print(TEST_YAML.read_text(encoding="utf-8"))
display(Markdown("### test_dataloader.yaml")); print(loader_path.read_text(encoding="utf-8"))


## 3. 実行前チェック


In [ ]:
files=sorted(data_dir.glob(loader_cfg["data"]["file_pattern"]))
expected={float(v) for v in loader_cfg["test_re"]}
found=set()
for path in files:
    try: found.add(float(path.stem.rsplit("_Re",1)[1]))
    except (IndexError,ValueError): pass
performance_root=(ROOT/"performance/data"/kind).resolve()
checks=[
 status(TEST_YAML.is_file(),"test config",TEST_YAML),
 status(loader_path.is_file(),"test dataloader config",loader_path),
 status(checkpoint.is_file(),"checkpoint",checkpoint),
 status(data_dir.is_dir(),"performance data directory",data_dir),
 status(data_dir.is_relative_to(performance_root),"data isolation",performance_root),
 status(bool(files),"pair files",f"{len(files)} files"),
 status(found==expected,"Re filenames",f"expected={sorted(expected)}, found={sorted(found)}"),
 status(result_dir.parents[1].name==EXPERIMENT_ID,"experiment result root",result_dir.parents[1]),
]
PRECHECK_OK=all(checks)
print("\n実行可能" if PRECHECK_OK else "\n設定またはデータを修正してください。")


## 4. 1-step評価の実行


In [ ]:
RUN_ONE_STEP = False
if RUN_ONE_STEP:
    if not PRECHECK_OK: raise RuntimeError("事前チェックが失敗しています。")
    subprocess.run([sys.executable,str(ROOT/"performance/src/test.py"),"--config",str(TEST_YAML)],cwd=ROOT,check=True)
else: print("未実行。確認後にRUN_ONE_STEP=Trueへ変更してください。")


## 5. Metrics


In [ ]:
metrics_csv=result_dir/test_cfg["output"]["metrics_csv"]
if not metrics_csv.exists(): print("結果がありません:",metrics_csv)
else:
 with metrics_csv.open(encoding="utf-8",newline="") as f: rows=list(csv.DictReader(f))
 selected=[r for r in rows if r["scale"]=="physical" and r["region"]=="valid_region" and r["Re"]!="all"]
 print("Re      channel       MSE             Relative L2")
 for r in selected: print(f'{r["Re"]:>5}   {r["channel"]:>7}   {float(r["mse"]):.6e}   {float(r["relative_l2"]):.6f}')


## 6. 保存済みグラフ


In [ ]:
plot_key="interpolation_plot" if kind=="interpolation" else "extrapolation_plot"
plot_path=result_dir/test_cfg["output"][plot_key]
if plot_path.exists(): display(Image(filename=str(plot_path)))
else: print("PNGがありません:",plot_path)
